**1. Install Required Libraries**

In [6]:
!pip install faiss-cpu langchain-huggingface langchain-community pypdf

**2. The Ingestion Script**

This script performs Step 4 (Retrieval) by creating embeddings of your documents. **bold text**

In [7]:
!pip install langchain-text-splitters

import os
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load Documents (Step 2: Knowledge Base)
# Place all your agricultural PDFs in a folder named 'knowledge_data'
loader = DirectoryLoader('./knowledge_data', glob="./*.pdf", loader_cls=PyPDFLoader)
documents = loader.load()

# 2. Split Text into Chunks
# Small chunks help the LLM find precise facts
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
text_chunks = text_splitter.split_documents(documents)

# 3. Create Vector Embeddings (Step 4: Retrieval)
# This turns human text into mathematical vectors
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. Build and Save the FAISS Index
vector_store = FAISS.from_documents(text_chunks, embeddings)
vector_store.save_local("faiss_agri_index")

print(f"Successfully created FAISS index with {len(text_chunks)} data chunks.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully created FAISS index with 6043 data chunks.


**3. Save the "Brain" to Google Drive**

Just like your models, you must save the FAISS index to Drive so your Django server can access it.

In [8]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/AgriBot_Project/VectorStore/'

if not os.path.exists(drive_path):
    os.makedirs(drive_path)

# Move the FAISS folder to Drive
shutil.copytree("faiss_agri_index", os.path.join(drive_path, "faiss_agri_index"), dirs_exist_ok=True)
print("Knowledge Base (FAISS Index) saved to Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Knowledge Base (FAISS Index) saved to Google Drive.


**🔍 Module: Retrieval Testing Script**

Goal: Verify that the FAISS index can locate factual evidence before we pass it to the LLM (Gemini).

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load the "Brain" (FAISS Index) you just created
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.load_local("faiss_agri_index", embeddings, allow_dangerous_deserialization=True)

# 2. Define a Test Query (Step 1: Input)
test_query = "What is the recommended nitrogen dosage for wheat?"

# 3. Perform Retrieval (Step 4: FAISS Search)
# We ask for the top 3 most relevant "chunks" of text
docs = vector_db.similarity_search(test_query, k=3)

# 4. Analyze the Results (Step 15: Analysis)
print(f"--- RETRIEVAL RESULTS FOR: '{test_query}' ---\n")
for i, doc in enumerate(docs):
    print(f"Result {i+1} (Source: {doc.metadata.get('source')}):")
    print(f"{doc.page_content}\n" + "-"*30)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- RETRIEVAL RESULTS FOR: 'What is the recommended nitrogen dosage for wheat?' ---

Result 1 (Source: knowledge_data/farmerbook.pdf):
Fertilizer doses 
Fertilizers /Age of 
tree
I  Ye a r II Y ear III Y ear IV Y ear and 
Above
Nitrogen 150 300 450 600
Phosphorus 50 100 150 200
Potassium 25 50 75 100
As far as possible 1/3rd of the dose of N may be given through farm yard manure/compost, oil cakes 
etc. 
Nagpur Mandarin
Disease Free Bud Grafts of Nagpur Mandarin
------------------------------
Result 2 (Source: knowledge_data/Training-Manual-English.pdf):
Empowering Farmers: Natural Farming Training Toolkit and Best Practices Guide
143
8.22 Wheat47  
Fig 8.22: Wheat Crop
(i) Land Preparation
• Along with the available farmyard manure, mix and apply 250 kg Ghanjeevamrit 
per	hectare	into	the	soil	during	field	preparation.	
(ii) Varieties
• For timely sowing: Lok-1, GW-366, GW-322, GW-496, GW-451, GW-503, 
GW-190, GW-273
• For limited irrigation: GW-1139, GW-1255, HI-8489
(iii) Seed Rate 